In [1]:

from __future__ import annotations

import argparse
import html
import logging
import re
import sys
from dataclasses import dataclass, field, fields
from datetime import date
from pathlib import Path
from typing import Optional
import xml.etree.ElementTree as ET

logger = logging.getLogger("analisis_riesgo")

CONCEPTOS_BALANCE: dict[str, list[str]] = {
    "pasivos_corrientes":             ["CurrentLiabilities"],
    "deuda_financiera_corto_plazo":   ["ObligacionesFinancierasCorrientes", "CurrentBorrowings"],
    "deuda_financiera_largo_plazo":   ["ObligacionesFinancierasNoCorrientes", "NoncurrentBorrowings"],
    "bonos_emitidos":                 ["TitulosEmitidos"],
}

# El activo corriente se reconstruye sumando estas partidas individuales.
PARTES_ACTIVO_CORRIENTE: list[str] = [
    "TradeAndOtherCurrentReceivables",
    "Inventories",
    "InversionesCorrientes",
    "OtherCurrentFinancialAssets",
    "OtherCurrentNonfinancialAssets",
    "CurrentTaxAssetsCurrent",
    "CashAndCashEquivalents",
]

CONCEPTOS_ANO: dict[str, list[str]] = {
    "ingresos":                 ["Revenue"],
    "utilidad_operacional":     ["ProfitLossFromOperatingActivities"],
    "gastos_financieros":       ["FinanceCosts"],
    "dep_amort":                ["AdjustmentsForDepreciationAndAmortisationExpense", "DepreciationAndAmortisationExpense"],
    "interes_pagado_caja":      ["InterestPaidClassifiedAsOperatingActivities", "InterestPaidClassifiedAsFinancingActivities"],
}


# ==============================================================================
# VALORES VERIFICADOS MANUALMENTE CONTRA EL PDF DEL ESTADO FINANCIERO
# ==============================================================================
# Casos donde el XBRL no trae el dato de forma estructurada (o lo trae en
# cero en todos los contextos) y se verificó el valor exacto contra el
# reporte en PDF publicado por el emisor.
VALORES_VERIFICADOS: dict[str, dict[str, float]] = {
    "0052249427_0260_000033_0000_000000_000000_C-C_2019-12-31.xbrl": {
        # EPM 2019, Estado del Resultado Integral Consolidado, p.11: no se
        # reportó subtotal de utilidad operacional en el XBRL. Calculado
        # como Total ingresos (18,359,614) - Costos por prestación de
        # servicios (11,557,807) - Gastos de administración (1,596,792) -
        # Deterioro de cuentas por cobrar (77,801) - Otros gastos
        # (157,467) = 4,969,747.
        "utilidad_operacional": 4_969_747_000,
    },
    "0054026625_0261_000034_0000_000000_000000_C-C_2024-12-31.xbrl": {
        # EMGESA/Enel Colombia 2024. Estado de Resultados Consolidado por
        # Naturaleza, Nota 28: Depreciaciones 887,169,956 + Amortizaciones
        # 241,202,013 = 1,128,371,969. El XBRL trae este campo en 0 en
        # todos los contextos del archivo.
        "dep_amort": 1_128_371_969,
    },
    "0011012534_0261_000034_0000_000000_000000_C-C_2015-12-31.xbrl": {
       
        "dep_amort": 164_215_881,
    },
    "0051216987_0261_000034_0000_000000_000000_C-C_2016-12-31.xbrl": {
        "dep_amort": 191_943_036,
    },
    "0051550243_0261_000034_0000_000000_000000_C-C_2017-12-31.xbrl": {
        "dep_amort": 210_447_724,
    },
    "0051903933_0261_000034_0000_000000_000000_C-C_2018-12-31.xbrl": {
        "dep_amort": 216_460_755,
    },
    "0052988197_0261_000034_0000_000000_000000_C-C_2021-12-31.xbrl": {
        "dep_amort": 247_319_743,
    },
    "0052247325_0261_000034_0000_000000_000000_C-C_2019-12-31.xbrl": {
        "dep_amort": 242_230_877,
    },
    "0052616283_0261_000034_0000_000000_000000_C-C_2020-12-31.xbrl": {
        "dep_amort": 245_505_363,
    },
    "0053345795_0261_000034_0000_000000_000000_C-C_2022-12-31.xbrl": {
        # Enel Colombia 2022 (Nota 28); salto frente a 2021 por la fusión
        # Emgesa-Codensa.
        "dep_amort": 859_900_474,
    },
    "0053682722_0261_000034_0000_000000_000000_C-C_2023-12-31.xbrl": {
        "dep_amort": 1_028_988_218,
    },
    "0054375223_0261_000034_0000_000000_000000_C-C_2025-12-31.xbrl": {

        "dep_amort": 1_146_676_800,
    },
}


# ==============================================================================
# CORRECCIONES DE ESCALA
# ==============================================================================
# Archivos donde el emisor reportó todas sus cifras en una unidad
# distinta . Previo a 2020, reportado en millones, posteriormente
# publicado en miles de millones.
# CELSIA 2015:El error se cancela en los 4 ratios finales 
#(afecta numerador y denominador por igual), pero se corrige para que las
#cifras absolutas en pesos no muestren un salto artificial frente a los demás años.

CORRECCIONES_DE_ESCALA: dict[str, float] = {
    "0011013677_0261_000026_0000_000000_000000_C-C_2015-12-31.xbrl": 1_000,
    "0052608752_0261_000026_0000_000000_000000_C-C_2020-12-31.xbrl": 1_000,
    "0052981101_0261_000026_0000_000000_000000_C-C_2021-12-31.xbrl": 1_000,
    "0053341484_0261_000026_0000_000000_000000_C-C_2022-12-31.xbrl": 1_000,
    "0053683977_0261_000026_0000_000000_000000_C-C_2023-12-31.xbrl": 1_000,
    "0054026672_0261_000026_0000_000000_000000_C-C_2024-12-31.xbrl": 1_000,
    "0054363600_0261_000026_0000_000000_000000_C-C_2025-12-31.xbrl": 1_000,
}

@dataclass
class DatosEmpresaAno:
    archivo: str
    nombre_entidad: Optional[str] = None
    fecha_balance: Optional[str] = None
    periodo_fin: Optional[str] = None

    pasivos_corrientes: Optional[float] = None
    activos_corrientes: Optional[float] = None
    deuda_financiera_corto_plazo: Optional[float] = None
    deuda_financiera_largo_plazo: Optional[float] = None
    bonos_emitidos: Optional[float] = None

    ingresos: Optional[float] = None
    utilidad_operacional: Optional[float] = None
    gastos_financieros: Optional[float] = None
    dep_amort: Optional[float] = None
    interes_pagado_caja: Optional[float] = None

    deuda_financiera_total: Optional[float] = None
    ebitda: Optional[float] = None
    ratio_deuda_ebitda: Optional[float] = None
    margen_ebitda: Optional[float] = None
    razon_corriente: Optional[float] = None
    cobertura_intereses: Optional[float] = None

    notas: list[str] = field(default_factory=list)

def _sin_prefijo(tag: str) -> str:
    """Quita el namespace de un tag XML: '{http://...}Assets' -> 'Assets'."""
    return tag.split("}")[-1]


def _leer_fechas(raiz: ET.Element) -> dict[str, dict]:
    """
    Devuelve {id_de_contexto: info} para todos los contextos del archivo.

    info['es_desglose'] es True si el contexto define una dimensión
    (<segment> o <scenario> con explicitMember/typedMember), lo que indica
    que es un desglose de nota (por segmento de negocio, filial, tipo de
    reserva, etc.) y no el dato principal del estado financiero. Se detecta
    por la presencia de <segment>/<scenario>, no por el nombre del
    contexto, ya que el nombre puede venir en distintos idiomas según el
    emisor (p.ej. ISA usa "Miembro" en vez de "Member").
    """
    fechas: dict[str, dict] = {}
    for el in raiz.iter():
        if _sin_prefijo(el.tag) != "context":
            continue
        cid = el.attrib.get("id")
        info = {"instante": None, "inicio": None, "fin": None, "es_desglose": False}
        for hijo in el:
            tag_hijo = _sin_prefijo(hijo.tag)
            if tag_hijo == "period":
                for dato in hijo:
                    tipo = _sin_prefijo(dato.tag)
                    if tipo == "instant":
                        info["instante"] = dato.text
                    elif tipo == "startDate":
                        info["inicio"] = dato.text
                    elif tipo == "endDate":
                        info["fin"] = dato.text
            elif tag_hijo == "entity":
                for nieto in hijo:
                    if _sin_prefijo(nieto.tag) == "segment":
                        info["es_desglose"] = True
            elif tag_hijo == "scenario":
                info["es_desglose"] = True
        fechas[cid] = info
    return fechas


def _dias_entre(inicio: str, fin: str) -> Optional[int]:
    try:
        d1 = date.fromisoformat(inicio[:10])
        d2 = date.fromisoformat(fin[:10])
        return (d2 - d1).days
    except (ValueError, TypeError):
        return None


def _elegir_contextos_principales(fechas: dict[str, dict]) -> tuple[Optional[str], Optional[str]]:
    """
    Selecciona el contexto de balance (fecha puntual más reciente) y el
    contexto de resultados de año completo (rango de 360-366 días con
    fecha de cierre más reciente), excluyendo contextos de desglose.

    Se restringe el rango de resultados a 360-366 días, y entre los
    candidatos que comparten la misma fecha de cierre se prioriza el de
    mayor duración. Esto evita que un contexto de un solo trimestre, que
    coincide en fecha de cierre con el año completo, sea seleccionado por
    error (caso detectado en EMGESA).
    """
    fecha_balance, valor_balance = None, None
    fecha_ano, valor_fin_ano, dias_ano = None, None, -1

    for cid, info in fechas.items():
        if info.get("es_desglose"):
            continue
        if info["instante"]:
            if valor_balance is None or info["instante"] > valor_balance:
                fecha_balance, valor_balance = cid, info["instante"]
        if info["inicio"] and info["fin"]:
            dias = _dias_entre(info["inicio"], info["fin"])
            if dias is None:
                continue
            es_un_ano = 360 <= dias <= 366
            if not es_un_ano:
                continue
            es_mas_reciente = valor_fin_ano is None or info["fin"] > valor_fin_ano
            es_mismo_cierre_pero_mas_largo = info["fin"] == valor_fin_ano and dias > dias_ano
            if es_mas_reciente or es_mismo_cierre_pero_mas_largo:
                fecha_ano, valor_fin_ano, dias_ano = cid, info["fin"], dias

    return fecha_balance, fecha_ano


def _extraer_datos_de_fecha(raiz: ET.Element, id_fecha: str) -> dict[str, str]:
    """Devuelve {nombre_concepto: valor_texto} para un contexto específico."""
    datos: dict[str, str] = {}
    for el in raiz:
        if el.attrib.get("contextRef") != id_fecha:
            continue
        nombre = _sin_prefijo(el.tag)
        valor = (el.text or "").strip()
        if valor:
            datos[nombre] = valor
    return datos


def _buscar_en_todo_el_archivo(raiz: ET.Element, nombres_posibles: list[str]) -> Optional[str]:
    """
    Busca un dato independiente del período (como el nombre de la entidad)
    en cualquier contexto del archivo, no solo en fecha_balance/fecha_ano.

    Necesario porque en varios emisores (p.ej. EMGESA/Enel Colombia) el
    nombre de la entidad vive en un contexto de portada que no coincide
    con ninguno de los dos contextos usados para los datos financieros.

    Los alias se recorren en orden de prioridad en el bucle externo, no en
    el orden en que aparecen en el documento: en CELSIA, por ejemplo,
    NameOfUltimateParentOfGroup ("Grupo Argos S.A.") aparece antes en el
    XML que NameOfReportingEntityOrOtherMeansOfIdentification ("CELSIA
    COLOMBIA S.A. E.S.P."), así que recorrer por orden de aparición
    devolvería el nombre de la matriz en vez de la propia empresa.
    """
    for nombre_tag in nombres_posibles:
        for el in raiz:
            if _sin_prefijo(el.tag) == nombre_tag:
                valor = (el.text or "").strip()
                if valor:
                    return _limpiar_texto_entidad(valor)
    return None


def _limpiar_texto_entidad(texto: str) -> str:
    """
    Elimina marcado HTML y decodifica entidades HTML del nombre de la
    entidad (caso detectado en EPM 2015, donde el campo viene como bloque
    de texto enriquecido en vez de texto plano).
    """
    if "<" in texto and ">" in texto:
        texto = re.sub(r"<[^>]+>", " ", texto)
    texto = html.unescape(texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def _a_numero(texto: str) -> Optional[float]:
    try:
        return float(texto)
    except (ValueError, TypeError):
        return None


def _buscar_primero(datos: dict[str, str], nombres_posibles: list[str]) -> Optional[float]:
    """
    Devuelve el primer valor no-cero entre los alias de la lista.

    Si el primer alias existe pero vale 0 porque ese año el concepto se
    reportó bajo una clasificación alternativa (caso ISA: el interés
    pagado aparece bajo "Operativo" con valor 0 mientras el valor real
    está en "Financiación"), no se acepta ese 0 de inmediato: se revisan
    todos los alias en busca de un valor distinto de cero, y solo si
    ninguno lo es, se acepta el cero como valor real.
    """
    valor_cero_encontrado = None
    for nombre in nombres_posibles:
        if nombre in datos:
            numero = _a_numero(datos[nombre])
            if numero is None:
                continue
            if numero != 0:
                return numero
            if valor_cero_encontrado is None:
                valor_cero_encontrado = numero
    return valor_cero_encontrado


def leer_archivo(ruta: Path) -> DatosEmpresaAno:
    """Lee un archivo .xbrl y devuelve todos los datos que pudo encontrar."""
    resultado = DatosEmpresaAno(archivo=ruta.name)

    try:
        arbol = ET.parse(ruta)
    except ET.ParseError as error:
        resultado.notas.append(f"Archivo XML inválido: {error}")
        return resultado

    raiz = arbol.getroot()
    fechas = _leer_fechas(raiz)
    fecha_balance, fecha_ano = _elegir_contextos_principales(fechas)

    if fecha_balance is None:
        resultado.notas.append("No se encontró la fecha del balance.")
    if fecha_ano is None:
        resultado.notas.append("No se encontró el rango del año de resultados.")

    datos_balance = _extraer_datos_de_fecha(raiz, fecha_balance) if fecha_balance else {}
    datos_ano = _extraer_datos_de_fecha(raiz, fecha_ano) if fecha_ano else {}

    resultado.fecha_balance = fechas.get(fecha_balance, {}).get("instante")
    resultado.periodo_fin = fechas.get(fecha_ano, {}).get("fin")

    resultado.nombre_entidad = _buscar_en_todo_el_archivo(
        raiz,
        [
            "NameOfReportingEntityOrOtherMeansOfIdentification",
            "NameOfParentEntity",
            "NameOfUltimateParentOfGroup",
        ],
    )

    for nombre_es, alias in CONCEPTOS_BALANCE.items():
        valor = _buscar_primero(datos_balance, alias)
        if valor is None:
            resultado.notas.append(f"No encontrado: {nombre_es}")
        setattr(resultado, nombre_es, valor)

    suma = 0.0
    encontro_algo = False
    for tag in PARTES_ACTIVO_CORRIENTE:
        if tag in datos_balance:
            numero = _a_numero(datos_balance[tag])
            if numero is not None:
                suma += numero
                encontro_algo = True
    resultado.activos_corrientes = suma if encontro_algo else None
    if not encontro_algo:
        resultado.notas.append("No se pudo reconstruir activos_corrientes.")

    for nombre_es, alias in CONCEPTOS_ANO.items():
        if nombre_es == "interes_pagado_caja":
            # El estándar XBRL permite reportar el interés pagado en
            # efectivo repartido entre dos secciones del flujo de caja
            # (Operativo y Financiación) simultáneamente. Se detectó en ISA
            # 2017-2020 que ambas secciones traían valor distinto de cero
            # el mismo año (p.ej. 2019: Operativo=208,505,151 y
            # Financiación=946,532,189), por lo que el pago real total es
            # la suma de ambas, no el primer valor no-cero. A diferencia de
            # _buscar_primero (donde los alias son nombres alternativos del
            # mismo dato bajo distintas taxonomías), aquí los dos alias son
            # componentes aditivos del mismo total.
            valores = [_a_numero(datos_ano[a]) for a in alias if a in datos_ano]
            valores_validos = [v for v in valores if v is not None]
            valor = sum(valores_validos) if valores_validos else None
        else:
            valor = _buscar_primero(datos_ano, alias)
        if valor is None:
            resultado.notas.append(f"No encontrado: {nombre_es}")
        setattr(resultado, nombre_es, valor)

    if ruta.name in VALORES_VERIFICADOS:
        for campo, valor in VALORES_VERIFICADOS[ruta.name].items():
            setattr(resultado, campo, valor)
            resultado.notas = [
                n for n in resultado.notas if n != f"No encontrado: {campo}"
            ]
            resultado.notas.append(f"{campo} = {valor:,} verificado a mano contra el PDF (ver VALORES_VERIFICADOS).")

    if ruta.name in CORRECCIONES_DE_ESCALA:
        factor = CORRECCIONES_DE_ESCALA[ruta.name]
        campos_numericos = list(CONCEPTOS_BALANCE.keys()) + ["activos_corrientes"] + list(CONCEPTOS_ANO.keys())
        for campo in campos_numericos:
            valor_actual = getattr(resultado, campo)
            if valor_actual is not None:
                setattr(resultado, campo, valor_actual * factor)
        resultado.notas.append(
            f"Todos los valores se multiplicaron x{factor:,.0f} (ver CORRECCIONES_DE_ESCALA): "
            f"el emisor reportó este archivo en la unidad equivocada."
        )

    return resultado


# ==============================================================================
# CÁLCULO DE RATIOS
# ==============================================================================

def calcular_ratios(d: DatosEmpresaAno) -> None:
    """Calcula los 4 ratios de riesgo y los asigna al objeto recibido."""

    # Deuda financiera total = deuda corto plazo + deuda largo plazo + bonos
    partes_deuda = [d.deuda_financiera_corto_plazo, d.deuda_financiera_largo_plazo, d.bonos_emitidos]
    if any(p is not None for p in partes_deuda):
        d.deuda_financiera_total = sum(p for p in partes_deuda if p is not None)

    # EBITDA = utilidad operacional + depreciación y amortización.
    #
    # Se detectó en EMGESA que el campo de D&A puede venir en 0 en todos
    # los contextos del archivo de que el emisor no desglosó el
    # dato en XBRL, no de que el D&A real sea cero. Se trata igual que un
    # dato faltante, dejando constancia en las notas, en vez de calcular un
    # EBITDA falsamente preciso.
    if d.utilidad_operacional is not None and d.dep_amort not in (None, 0):
        d.ebitda = d.utilidad_operacional + d.dep_amort
    elif d.utilidad_operacional is not None:
        d.ebitda = d.utilidad_operacional
        d.notas.append("EBITDA sin sumar D&A (dato faltante o en cero); es una aproximación.")

    # Deuda / EBITDA
    if d.deuda_financiera_total is not None and d.ebitda not in (None, 0):
        d.ratio_deuda_ebitda = d.deuda_financiera_total / d.ebitda

    # Margen EBITDA
    if d.ebitda is not None and d.ingresos not in (None, 0):
        d.margen_ebitda = d.ebitda / d.ingresos

    # Razón corriente
    if d.activos_corrientes is not None and d.pasivos_corrientes not in (None, 0):
        d.razon_corriente = d.activos_corrientes / d.pasivos_corrientes

    # Cobertura de intereses = EBITDA / Gastos financieros. Se prioriza el
    # interés pagado en efectivo sobre el gasto financiero contable por ser
    # más representativo del flujo real.
    #
    # Se detectó en EMGESA que el interés pagado en efectivo puede venir en
    # 0 en un año específico (clasificado bajo otra categoría de flujo de
    # caja ese año), sin que sea un dato ausente. Igual que en
    # _buscar_primero, solo se usa el interés en efectivo si es distinto de
    # cero; en caso contrario se usa el gasto financiero contable.
    gasto_interes = d.interes_pagado_caja if d.interes_pagado_caja not in (None, 0) else d.gastos_financieros
    if d.ebitda is not None and gasto_interes not in (None, 0):
        d.cobertura_intereses = d.ebitda / gasto_interes



def encontrar_archivos_xbrl(ruta: Path) -> list[Path]:
    if ruta.is_file():
        return [ruta]
    if ruta.is_dir():
        archivos = sorted(ruta.rglob("*.xbrl"))
        if not archivos:
            logger.warning("No se encontraron archivos .xbrl en %s", ruta)
        return archivos
    raise FileNotFoundError(f"No existe la ruta: {ruta}")


def procesar_todos(rutas: list[Path]) -> list[DatosEmpresaAno]:
    resultados = []
    for ruta in rutas:
        logger.info("Procesando: %s", ruta.name)
        datos = leer_archivo(ruta)
        calcular_ratios(datos)
        if datos.notas:
            logger.warning("  %s: %d nota(s) (%s)", ruta.name, len(datos.notas), datos.notas[0])
        resultados.append(datos)
    return resultados


def guardar_excel(resultados: list[DatosEmpresaAno], ruta_salida: Path) -> None:
    try:
        import openpyxl
    except ImportError:
        print("Falta instalar openpyxl. Ejecuta: pip install openpyxl")
        sys.exit(1)

    campos = [f.name for f in fields(DatosEmpresaAno) if f.name != "notas"] + ["notas"]

    libro = openpyxl.Workbook()
    hoja = libro.active
    hoja.title = "Ratios de riesgo"
    hoja.append(campos)

    for d in resultados:
        fila = []
        for campo in campos:
            valor = getattr(d, campo)
            fila.append(" | ".join(valor) if campo == "notas" else valor)
        hoja.append(fila)

    for i, campo in enumerate(campos, start=1):
        letra = openpyxl.utils.get_column_letter(i)
        hoja.column_dimensions[letra].width = max(16, min(32, len(campo) + 2))
    hoja.freeze_panes = "A2"

    libro.save(ruta_salida)
    logger.info("Guardado en: %s", ruta_salida)


def main(argv: Optional[list[str]] = None) -> int:
    parser = argparse.ArgumentParser(
        description="Calcula ratios de riesgo de incumplimiento a partir de archivos XBRL."
    )
    parser.add_argument("entrada", type=Path, help="Un archivo .xbrl o una carpeta con varios")
    parser.add_argument("-o", "--salida", type=Path, default=Path("ratios_riesgo.xlsx"),
                         help="Archivo Excel de salida (por defecto: ratios_riesgo.xlsx)")
    parser.add_argument("-v", "--verbose", action="store_true", help="Muestra más detalle en pantalla")
    argumentos = parser.parse_args(argv)

    logging.basicConfig(
        level=logging.DEBUG if argumentos.verbose else logging.INFO,
        format="%(levelname)s: %(message)s",
    )

    try:
        rutas = encontrar_archivos_xbrl(argumentos.entrada)
    except FileNotFoundError as error:
        logger.error(str(error))
        return 1

    if not rutas:
        return 1

    resultados = procesar_todos(rutas)
    guardar_excel(resultados, argumentos.salida)

    con_notas = sum(1 for r in resultados if r.notas)
    logger.info(
        "Listo: %d archivo(s) procesado(s), %d con notas (revisa la columna 'notas' del Excel).",
        len(resultados), con_notas,
    )
    return 0


In [2]:

"""
Convierte los 4 ratios de riesgo crediticio ya calculados (por
analisis_riesgo_crediticio.py, en resultado_completo.xlsx) en un scorecard
de riesgo por empresa-año:

  1. Se puntúa cada ratio de 1 a 5 según umbrales fijos, con peso igual (25% cada uno).
  2. Se promedia en un score_compuesto y se clasifica en una banda_riesgo.
  3. Se agrega una columna de tendencia frente al año anterior de la misma
     empresa.
  4. Se cruza cada empresa-año contra una tabla de eventos cualitativos
     reales (cambios de calificación, fusiones, intervenciones), documentados
     a partir de reportes de Fitch, BRC y prensa.
"""

from datetime import datetime

ENTRADA = "resultado_completo.xlsx"
SALIDA = "scorecard_riesgo.xlsx"


# ==============================================================================
# EVENTOS CUALITATIVOS DOCUMENTADOS
# ==============================================================================
# Descripción del evento y su efecto
# en la calificación, con la fuente.
EVENTOS = {
    ("EPM", 2018): "Colapso del tunel de desviacion de Hidroituango -> Observacion Negativa, baja BBB+ a BBB (Fitch)",
    ("EPM", 2020): "Intervencion politica en la junta directiva (Alcaldia Medellin) -> baja BBB a BBB- (Fitch)",
    ("EPM", 2021): "Efecto cascada: baja soberana de Colombia y Medellin -> baja BBB- a BB+ (Fitch)",
    ("EPM", 2025): "Cierre exitoso tunel Hidroituango, pero downgrade BB+ a BB por vencimientos de deuda 2026-27 y gobierno corporativo (Fitch, dic-2025)",
    ("ISA", 2016): "Upgrade internacional BBB a BBB+ por metricas crediticias fuertes vs pares regionales (Fitch)",
    ("ISA", 2021): "Downgrade internacional BBB+ a BBB por baja soberana de Colombia (Fitch, jul-2021)",
    ("ISAGEN", 2016): "Venta de participacion mayoritaria a Brookfield (cerrada ene-2016) -> outlook internacional sube a Positivo (Fitch)",
    ("ISAGEN", 2021): "Downgrade internacional BBB- a BB+ por perdida de grado de inversion soberano (S&P, may-2021)",
    ("ISAGEN", 2024): "Fusion por absorcion de activos solares de Matrix Renewables Colombia -> confirmado neutral para el rating (Fitch, feb-2024)",
    ("CELSIA", 2022): "Upgrade matriz Celsia S.A. de AA+ a AAA por mejora EBITDA +32% y menores perdidas de energia (BRC)",
    ("EMGESA-ENEL COLOMBIA", 2021): "Downgrade internacional BBB a BBB- por perdida de grado de inversion soberano (S&P, may-2021)",
    ("EMGESA-ENEL COLOMBIA", 2022): "Fusion Emgesa-Codensa-Enel Green Power -> Enel Colombia; vista positiva por Fitch (mayor escala/diversificacion), credit neutral para S&P",
    ("EMGESA-ENEL COLOMBIA", 2025): "Downgrade internacional BBB a BBB- por baja del soberano colombiano (Fitch, dic-2025)",
}

def empresa_canonica(nombre):
    """
    Mapea el nombre de entidad tal como aparece en el XBRL (que varía entre
    razón social completa, sigla, o con/sin tildes según el emisor y el año)
    a un nombre canónico único por empresa, para poder agrupar y cruzar con
    EVENTOS de forma consistente.
    """
    n = (nombre or "").upper()
    n = (n.replace("Ó", "O").replace("É", "E").replace("Á", "A")
          .replace("Í", "I").replace("Ú", "U"))
    if "EPM" in n:
        return "EPM"
    if "ISAGEN" in n:
        return "ISAGEN"
    if "ISA" in n or "INTERCONEXION" in n:
        return "ISA"
    if "EPSA" in n or "CELSIA" in n:
        return "CELSIA"
    if "EMGESA" in n or "ENEL" in n:
        return "EMGESA-ENEL COLOMBIA"
    return nombre


# ==============================================================================
# FUNCIONES DE SCORE (1 = riesgo alto, 3 = riesgo moderado, 5 = riesgo bajo)
# ==============================================================================
# Los umbrales siguen la lógica de un scorecard tipo Altman Z-Score adaptado
# a utilities reguladas.

def score_deuda_ebitda(v):
    """Deuda financiera / EBITDA. Menor apalancamiento = menor riesgo."""
    if v is None:
        return None
    if v < 3.0:
        return 5
    if v <= 4.0:
        return 3
    return 1


def score_cobertura(v):
    """EBITDA / gastos financieros. Mayor cobertura = menor riesgo."""
    if v is None:
        return None
    if v > 4.25:
        return 5
    if v >= 2.5:
        return 3
    return 1


def score_corriente(v):
    """Activos corrientes / pasivos corrientes. Mayor liquidez = menor riesgo."""
    if v is None:
        return None
    if v > 1.2:
        return 5
    if v >= 0.8:
        return 3
    return 1


def score_margen(v):
    """EBITDA / ingresos. Mayor rentabilidad operativa = menor riesgo."""
    if v is None:
        return None
    if v > 0.25:
        return 5
    if v >= 0.15:
        return 3
    return 1


def banda(score):
    """Clasifica el score_compuesto (promedio de los 4 scores) en una banda."""
    if score is None:
        return None
    if score >= 4.0:
        return "Bajo riesgo"
    if score >= 2.5:
        return "Riesgo moderado"
    return "Riesgo alto"


import openpyxl

wb_in = openpyxl.load_workbook(ENTRADA)
ws_in = wb_in.active
headers = [c.value for c in ws_in[1]]
idx = {h: i for i, h in enumerate(headers)}

registros = []
for row in ws_in.iter_rows(min_row=2, max_row=ws_in.max_row):
    periodo_fin = row[idx["periodo_fin"]].value
    if isinstance(periodo_fin, datetime):
        anio = periodo_fin.year
    elif isinstance(periodo_fin, str) and len(periodo_fin) >= 4:
        anio = int(periodo_fin[:4])
    else:
        anio = None

    empresa = empresa_canonica(row[idx["nombre_entidad"]].value)
    r_deuda = row[idx["ratio_deuda_ebitda"]].value
    r_cob = row[idx["cobertura_intereses"]].value
    r_cor = row[idx["razon_corriente"]].value
    r_mar = row[idx["margen_ebitda"]].value

    s_deuda = score_deuda_ebitda(r_deuda)
    s_cob = score_cobertura(r_cob)
    s_cor = score_corriente(r_cor)
    s_mar = score_margen(r_mar)

    # Peso igual (25% cada uno). Si falta algún ratio ese año, se promedia
    # solo entre los disponibles en vez de descartar la fila completa.
    disponibles = [s for s in [s_deuda, s_cob, s_cor, s_mar] if s is not None]
    score_compuesto = round(sum(disponibles) / len(disponibles), 2) if disponibles else None
    banda_riesgo = banda(score_compuesto)
    evento = EVENTOS.get((empresa, anio))

    registros.append({
        "empresa": empresa, "anio": anio,
        "ratio_deuda_ebitda": r_deuda, "cobertura_intereses": r_cob,
        "razon_corriente": r_cor, "margen_ebitda": r_mar,
        "score_deuda_ebitda": s_deuda, "score_cobertura": s_cob,
        "score_corriente": s_cor, "score_margen": s_mar,
        "score_compuesto": score_compuesto, "banda_riesgo": banda_riesgo,
        "evento_cualitativo": evento,
    })

por_empresa = {}
for reg in registros:
    por_empresa.setdefault(reg["empresa"], []).append(reg)

for empresa, regs in por_empresa.items():
    regs_ordenados = sorted([r for r in regs if r["anio"] is not None], key=lambda x: x["anio"])
    anterior = None
    for reg in regs_ordenados:
        if anterior is None or reg["score_compuesto"] is None or anterior["score_compuesto"] is None:
            reg["tendencia"] = "Primer año"
        elif reg["score_compuesto"] > anterior["score_compuesto"]:
            reg["tendencia"] = "Mejora"
        elif reg["score_compuesto"] < anterior["score_compuesto"]:
            reg["tendencia"] = "Deterioro"
        else:
            reg["tendencia"] = "Estable"
        anterior = reg

wb_out = openpyxl.Workbook()
ws_out = wb_out.active
ws_out.title = "Scorecard"

columnas = ["empresa", "anio", "ratio_deuda_ebitda", "cobertura_intereses", "razon_corriente",
            "margen_ebitda", "score_deuda_ebitda", "score_cobertura", "score_corriente",
            "score_margen", "score_compuesto", "banda_riesgo", "tendencia", "evento_cualitativo"]
ws_out.append(columnas)

registros_ordenados = sorted(registros, key=lambda r: (r["empresa"], r["anio"] or 0))
for reg in registros_ordenados:
    ws_out.append([reg.get(c) for c in columnas])

wb_out.save(SALIDA)
print(f"Listo -> {SALIDA} ({len(registros)} filas)")

Listo -> scorecard_riesgo.xlsx (55 filas)
